In [56]:
from connect4.game import ConnectFour
from connect4.agents import RandomAgent, GreedyAgent
from connect4.arena import play_match
import torch
import torch.nn as nn

In [5]:
game = ConnectFour()
greedy_agent = GreedyAgent()
random_agent = RandomAgent()

In [8]:
play_match(greedy_agent, random_agent, 10)

{'wins': 9, 'draws': 0, 'losses': 1, 'wins_as_first': 4, 'wins_as_second': 5}

In [88]:
torch.zeros(1)

tensor([0.])

In [110]:
torch.manual_seed(0)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)           # registered as a submodule
        self.fc2 = nn.Linear(8, 10)
        self.scale = nn.Parameter(torch.ones(10))            # learnable tensor
        self.register_buffer("offset", torch.zeros(1))      # saved, but not trained

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x) * self.scale + self.offset

net = Net()
# 2. Inspect the module
net.train()


Net(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=10, bias=True)
)

In [ ]:
x = torch.randn(3, 4)                        # batch of 3

# 1. Forward pass (calls forward())
y = net(x)
print(y.shape)                               # torch.Size([3, 2])

# 2. Inspect the module
print(net)                                   # structure
for name, p in net.named_parameters():       # fc1.weight, fc1.bias, ..., scale
    print(name, tuple(p.shape), p.requires_grad)
print(sum(p.numel() for p in net.parameters()))   # parameter count
print(dict(net.named_buffers()))             # offset

# 3. state_dict (parameters + buffers), used for saving/loading
sd = net.state_dict()
print(list(sd.keys()))
net2 = Net()
net2.load_state_dict(sd)                     # net2 is now identical to net

# 4. Train / eval mode (matters for Dropout, BatchNorm)
net.eval()
print(net.training)                          # False
net.train()

# 5. Freeze a layer
for p in net.fc1.parameters():
    p.requires_grad = False

# 6. One training step
opt = torch.optim.SGD([p for p in net.parameters() if p.requires_grad], lr=0.1)
target = torch.randn(3, 2)
loss = nn.functional.mse_loss(net(x), target)
opt.zero_grad()
loss.backward()
opt.step()
print(loss.item())

# 7. Forward hook: peek at an intermediate output
net.fc1.register_forward_hook(lambda m, inp, out: print("fc1 out:", out.shape))
net(x)

fc1 out: torch.Size([3, 8])
torch.Size([3, 2])


tensor([[-0.1937, -0.2214],
        [ 0.1400, -0.0172],
        [ 0.0429, -0.3425]], grad_fn=<AddBackward0>)

In [107]:
torch.ones(1,10).squeeze(-1)

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [108]:
torch.ones(10,1).squeeze(-1)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])